# Agent Loop
# 0. 介绍

**研究背景**：大模型每次 API 调用只会根据当前消息生成一次回复。Agent 要完成需要多个步骤的任务，就必须由外层程序反复执行`模型决策`、`工具执行`和`结果写回`，直到任务完成或满足停止条件。这个控制循环就是 Agent Loop。

**现存问题**：如果程序只调用一次大模型，那么模型即使正确选择了工具，也无法继续读取工具结果并完成下一步。另一方面，如果循环没有明确的步数上限和停止条件，模型可能不断请求工具，造成重复执行、Token 浪费和任务失控。

**解决方案**：本 Notebook 将实现一个极简的 Agent Loop，把工具结果写回消息历史，再交给同一个真实 API 模型继续决策。然后使用相同的模型和两个多步任务进行对比：基线版本只执行一次决策，改进版本持续执行`决策 → 工具 → 回写`循环，并通过任务成功、模型自然结束或达到最大步数明确停止，从而直观看到外层循环如何把单次模型调用变成能够完成任务的 Agent。

## 目录
0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 说明可用工具
Agent 需要先知道自己可以做什么。本节准备两个工具：`read_record` 读取一条记录，`submit_answer` 提交最终答案。每个工具都用简单的 JSON Schema 说明需要哪些参数。

In [2]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_record",
            "description": "读取指定记录",
            "parameters": {
                "type": "object",
                "properties": {
                    "record_id": {"type": "string"},
                },
                "required": ["record_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "submit_answer",
            "description": "提交最终答案",
            "parameters": {
                "type": "object",
                "properties": {
                    "answer": {"type": "string"},
                },
                "required": ["answer"],
            },
        },
    },
]
for tool in tools:
    print(f"可用工具：{tool['function']['name']}")

可用工具：read_record
可用工具：submit_answer


输出显示两个工具已经准备好。大模型之后可以先读取记录，再提交读到的内容；此时只是写好了工具说明，还没有真正执行工具。

## 2.2 准备两个多步任务
为了观察循环是否真的工作，本节准备两条记录和两个任务。每个任务都必须先读取指定记录，再把读到的内容提交出去，因此一次模型回复无法走完整个过程。

In [3]:
records = {"city": "杭州", "number": "42"}
tasks = [
    {"record_id": "city", "question": "读取 city 记录并提交记录内容。"},
    {"record_id": "number", "question": "读取 number 记录并提交记录内容。"},
]
for task in tasks:
    print(f"任务：{task['question']}")

任务：读取 city 记录并提交记录内容。
任务：读取 number 记录并提交记录内容。


输出显示两个任务已经准备好。它们的步骤相同，但读取的记录不同，后面可以用它们检查同一个 Agent Loop 能否重复完成多步任务。

## 2.3 写出初始消息
每次任务开始时，大模型需要收到一条工作规则和一条具体任务。本节把这两条消息放进列表，并先为第一个任务生成初始消息。

In [4]:
def make_messages(task):
    return [
        {"role": "system", "content": "先读取记录，再提交读到的内容。每次只调用一个工具。"},
        {"role": "user", "content": task["question"]},
    ]

messages = make_messages(tasks[0])
for message in messages:
    print(f"{message['role']}：{message['content']}")

system：先读取记录，再提交读到的内容。每次只调用一个工具。
user：读取 city 记录并提交记录内容。


输出显示第一项任务的初始上下文只有两条消息：`system` 说明执行顺序，`user` 给出具体任务。后面的循环会在这个列表末尾继续加入模型操作和工具结果。

## 2.4 定义工具执行方式
大模型只能提出工具请求，真正的读取和提交由 Python 完成。本节把两个工具对应到最直接的操作：读取记录时返回内容，提交答案时把内容写入任务状态。

In [5]:
def execute_tool(name, arguments, state):
    if name == "read_record":
        return records[arguments["record_id"]]

    state["answer"] = arguments["answer"]
    return "答案已提交"

print("工具执行方式已准备好")

工具执行方式已准备好


输出说明工具执行函数已经定义，但还没有运行。之后每当大模型选择一个工具，程序就会把工具名称和参数交给这个函数。

## 2.5 定义完成条件
Agent Loop 必须知道什么时候结束。本节规定：任务状态中已经提交答案，而且答案与指定记录的内容相同，任务才算完成。

In [6]:
def task_finished(task, state):
    expected = records[task["record_id"]]
    return state.get("answer") == expected

print("完成条件：提交答案与记录内容相同")

完成条件：提交答案与记录内容相同


输出说明循环已经有了清楚的结束目标。到这里，模型、工具、任务、初始消息、执行方式和完成条件都已准备好；下一章将发送第一条真实 API 请求，查看模型做出的第一步决策。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
程序已经准备好模型、工具和第一项任务，现在可以把它们一起发给大模型。本节要求大模型必须选择一个工具，并记录从发出请求到收到回复所用的时间。

In [7]:
import json
from time import perf_counter

start = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",  # 必须选择一个工具
    temperature=0,
)
latency_ms = round((perf_counter() - start) * 1000)
raw_real = response.model_dump()
print(f"回复已收到：provider={config['NANO_BACKEND']}, model={model_name}, latency={latency_ms} ms")
print(json.dumps(raw_real, indent=4, ensure_ascii=False))

回复已收到：provider=openai, model=LongCat-2.0, latency=2848 ms
{
    "id": "4ff9534f588e43f2871567206f642c7c",
    "choices": [
        {
            "finish_reason": "tool_calls",
            "index": 0,
            "logprobs": null,
            "message": {
                "content": null,
                "refusal": null,
                "role": "assistant",
                "annotations": null,
                "audio": null,
                "function_call": null,
                "tool_calls": [
                    {
                        "id": "call_9321970fdc5c441888190b17",
                        "function": {
                            "arguments": "{\"record_id\": \"city\"}",
                            "name": "read_record"
                        },
                        "type": "function",
                        "index": null
                    }
                ],
                "reasoning_content": "\n用户要求我读取 city 记录并提交记录内容。根据指示，我需要先读取记录，然后提交读到的内容。每次只调用一个工具。\n\n首先，我需要读取 

输出显示了模型来源、模型名称和等待时间，说明真实大模型已经返回第一步决策。完整结果保存在 `raw_real` 中，下一节会从中取出模型选择的工具和参数。

## 3.2 查看并保存响应
大模型已经返回结果，但长段原始数据不容易阅读。本节取出工具名称、参数、停止原因和 Token 用量，并把模型消息接到初始消息后面。

In [8]:
choice = raw_real["choices"][0]
assistant_message = choice["message"]
tool_call = assistant_message["tool_calls"][0]
arguments = json.loads(tool_call["function"]["arguments"])
real_messages = messages + [assistant_message]
print(f"工具：{tool_call['function']['name']}")
print(f"参数：{arguments}")
print(f"停止原因：{choice['finish_reason']}")
print(f"Token 用量：{raw_real['usage']}")
print(f"消息数量：{len(real_messages)}")

工具：read_record
参数：{'record_id': 'city'}
停止原因：tool_calls
Token 用量：{'completion_tokens': 61, 'prompt_tokens': 205, 'total_tokens': 266, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 43, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': 0, 'video_tokens': 0, 'text_tokens': 0}, 'effectiveCachedTokens': 128, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'output_tokens_details': None, 'cached_tokens': 0}
消息数量：3


输出显示大模型选择了 `read_record` 并填写记录编号，说明模型已经做出正确的第一步决策。`real_messages` 现在依次保存 system、user 和 assistant 消息；下一章将定义一个只执行这一步就停止的基线组件。

# 4. 定义基线组件
## 4.1 定义单步执行器
最简单的 Agent 只请求模型一次，执行模型选出的第一个工具，然后马上结束。这个版本不会把工具结果交回模型，正好用来说明没有循环时任务为什么无法走完。

In [9]:
def baseline_once(task):
    state = {"answer": None}
    task_messages = make_messages(task)
    response = client.chat.completions.create(
        model=model_name, messages=task_messages, tools=tools,
        tool_choice="required", temperature=0,
    )
    assistant = response.choices[0].message.model_dump(exclude_none=True)
    call = assistant["tool_calls"][0]
    args = json.loads(call["function"]["arguments"])
    result = execute_tool(call["function"]["name"], args, state)
    task_messages.append(assistant)
    return {"state": state, "tool_result": result,
            "stop_reason": "single_step", "messages": task_messages}

print("基线组件已准备好：每个任务只走一步")

基线组件已准备好：每个任务只走一步


输出说明单步执行器已经定义完成，但本节还没有运行任务。下一章会用两个真实任务调用它，查看工具虽然执行了，最终答案为什么仍然没有提交。

# 5. 展示基线故障
## 5.1 运行两个任务
现在让单步执行器分别处理两个任务。每个任务都会调用一次真实大模型并执行模型选择的工具，同时记录两项任务的总等待时间。

In [10]:
baseline_results = []
start = perf_counter()

for task in tasks:
    result = baseline_once(task)
    baseline_results.append(result)

baseline_latency_ms = round((perf_counter() - start) * 1000)
print(f"真实 API 调用：{len(baseline_results)} 次")
print(f"总等待时间：{baseline_latency_ms} ms")

真实 API 调用：2 次
总等待时间：4871 ms


输出说明两个任务都完成了一次真实 API 调用。此时只能确定单步执行器已经运行，下一节还要打开每项结果，查看工具返回值是否变成了最终答案。

## 5.2 查看单步结果
单步执行器在工具返回后立即结束。本节逐项打印工具结果、消息数量和任务状态，直接观察数据停在了哪里。

In [11]:
for task, result in zip(tasks, baseline_results):
    print(f"任务：{task['record_id']}")
    print(f"工具结果：{result['tool_result']}")
    print(f"消息数量：{len(result['messages'])}")
    print(f"最终答案：{result['state']['answer']}")
    print()

任务：city
工具结果：杭州
消息数量：3
最终答案：None

任务：number
工具结果：42
消息数量：3
最终答案：None



输出显示工具已经读到正确内容，但每项任务都只有三条消息，最终答案仍为空。数据停在了工具结果这里，因为单步执行器没有把结果写回消息，也没有再次请求模型。

## 5.3 判断任务结果
看见空答案还不够直观。本节使用第 2 章定义的同一个完成条件，判断两个任务是否真正提交了各自记录中的内容。

In [12]:
baseline_passed = []

for task, result in zip(tasks, baseline_results):
    passed = task_finished(task, result["state"])
    baseline_passed.append(passed)
    print(f"{task['record_id']} 任务通过：{passed}")

baseline_success_rate = sum(baseline_passed) / len(baseline_passed)
print(f"基线成功率：{baseline_success_rate:.0%}")

city 任务通过：False
number 任务通过：False
基线成功率：0%


输出中的两个 `False` 和 0% 成功率说明基线没有完成任何任务。模型已经正确读取记录，真正缺少的是把工具结果交回模型并继续下一步的循环；下一章将定义这个改进组件。

# 6. 定义改进组件
## 6.1 定义 Agent Loop
基线执行一次工具后就结束，所以模型看不到工具返回了什么。Agent Loop 会保留同一份消息和任务状态，每轮依次完成`请求模型 → 保存模型操作 → 执行工具 → 写回工具结果`。任务完成、模型不再调用工具或步数用完时，循环结束并说明原因。

In [13]:
def agent_loop(task, max_steps=3):
    state = {"answer": None}
    task_messages = make_messages(task)
    total_tokens = 0
    steps = 0
    stop_reason = "max_steps"

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=model_name,
            messages=task_messages,
            tools=tools,
            temperature=0,
        )
        total_tokens += response.usage.total_tokens
        steps = step

        # 模型操作也要进入消息历史
        assistant = response.choices[0].message.model_dump(
            exclude_none=True
        )
        task_messages.append(assistant)

        tool_calls = assistant.get("tool_calls", [])
        if not tool_calls:
            stop_reason = "model_finished"
            break

        # Python 执行模型选择的工具
        call = tool_calls[0]
        tool_name = call["function"]["name"]
        arguments = json.loads(call["function"]["arguments"])
        tool_result = execute_tool(tool_name, arguments, state)

        # 工具结果写回后，下一轮模型才能看到它
        tool_message = {
            "role": "tool",
            "tool_call_id": call["id"],
            "content": tool_result,
        }
        task_messages.append(tool_message)

        if task_finished(task, state):
            stop_reason = "success"
            break

    return {
        "state": state,
        "messages": task_messages,
        "steps": steps,
        "tokens": total_tokens,
        "stop_reason": stop_reason,
    }

print("改进组件已准备好：工具结果会写回消息并继续下一步")

改进组件已准备好：工具结果会写回消息并继续下一步


输出说明 Agent Loop 已经定义完成，但还没有运行任务。循环中的 `task_messages` 保存每轮输入和输出，`state` 保存最终答案，`stop_reason` 说明循环为什么结束。下一章将把与基线相同的两个任务交给它。

# 7. 展示修复结果
## 7.1 运行两个任务
为了只比较有无循环，本节把基线使用过的两个任务交给 Agent Loop。程序记录真实 API 调用次数、Token 总量和总等待时间，方便看清多轮执行的实际开销。

In [14]:
loop_results = []
loop_api_calls = 0
loop_tokens = 0
start = perf_counter()

for task in tasks:
    result = agent_loop(task)
    loop_results.append(result)
    loop_api_calls += result["steps"]
    loop_tokens += result["tokens"]

loop_latency_ms = round((perf_counter() - start) * 1000)
print(f"provider：{config['NANO_BACKEND']}")
print(f"model：{model_name}")
print(f"真实 API 调用：{loop_api_calls} 次")
print(f"Token 总量：{loop_tokens}")
print(f"总等待时间：{loop_latency_ms} ms")

provider：openai
model：LongCat-2.0
真实 API 调用：4 次
Token 总量：1232
总等待时间：8898 ms


输出显示两个任务都经过了多轮真实 API 调用，并记录了对应的 Token 和等待时间。下一节会展开第一项任务的消息历史，查看这些调用之间传递了什么。

## 7.2 查看消息流
只看调用次数仍然看不出循环怎样工作。本节按顺序打印第一项任务的消息：普通消息显示内容，模型消息显示工具调用，工具消息显示执行结果。

In [15]:
city_messages = loop_results[0]["messages"]

for number, message in enumerate(city_messages, start=1):
    role = message["role"]
    print(f"{number}. {role}")

    if role == "assistant":
        call = message["tool_calls"][0]["function"]
        print(f"   调用：{call['name']} {call['arguments']}")
    else:
        print(f"   内容：{message['content']}")

1. system
   内容：先读取记录，再提交读到的内容。每次只调用一个工具。
2. user
   内容：读取 city 记录并提交记录内容。
3. assistant
   调用：read_record {"record_id": "city"}
4. tool
   内容：杭州
5. assistant
   调用：submit_answer {"answer": "杭州"}
6. tool
   内容：答案已提交


输出中的消息顺序展示了完整闭环：用户提出任务，模型调用 `read_record`，工具返回记录内容，模型看到结果后调用 `submit_answer`，工具确认提交。基线缺少的正是中间的结果写回和下一轮决策。

## 7.3 判断任务结果
消息流已经走完，还要查看最终产物。本节使用与基线相同的完成条件，逐项显示提交答案、执行步数、停止原因和是否完成。

In [16]:
loop_passed = []

for task, result in zip(tasks, loop_results):
    passed = task_finished(task, result["state"])
    loop_passed.append(passed)
    print(f"任务：{task['record_id']}")
    print(f"最终答案：{result['state']['answer']}")
    print(f"执行步数：{result['steps']}")
    print(f"停止原因：{result['stop_reason']}")
    print(f"任务通过：{passed}")
    print()

loop_success_rate = sum(loop_passed) / len(loop_passed)
print(f"Agent Loop 成功率：{loop_success_rate:.0%}")

任务：city
最终答案：杭州
执行步数：2
停止原因：success
任务通过：True

任务：number
最终答案：42
执行步数：2
停止原因：success
任务通过：True

Agent Loop 成功率：100%


输出中的两个 `True` 和 100% 成功率说明 Agent Loop 完成了两项任务。模型、工具和完成条件都没有改变，唯一变化是工具结果被写回消息并触发下一轮决策；下一章将汇总基线与改进结果。

# 8. 汇总消融对照
## 8.1 对比两种做法
只改变有无 Agent Loop，再并排比较结果，就能看出循环是否必要。本节汇总共同使用的模型和任务，再展示两种做法的 API 调用、等待时间、最终答案和成功率。

In [17]:
baseline_answers = []
for result in baseline_results:
    baseline_answers.append(result["state"]["answer"])

loop_answers = []
for result in loop_results:
    loop_answers.append(result["state"]["answer"])

shared_info = {
    "模型来源": config["NANO_BACKEND"],
    "模型": model_name,
    "任务数量": len(tasks),
}
comparison = [
    {"做法": "单步执行", "API 调用": len(baseline_results), "等待时间（毫秒）": baseline_latency_ms, "最终答案": baseline_answers, "成功率": f"{baseline_success_rate:.0%}"},
    {"做法": "Agent Loop", "API 调用": loop_api_calls, "等待时间（毫秒）": loop_latency_ms, "最终答案": loop_answers, "成功率": f"{loop_success_rate:.0%}"},
]

print("共同信息：")
print(json.dumps(shared_info, ensure_ascii=False, indent=2))
print("消融对照：")
print(json.dumps(comparison, ensure_ascii=False, indent=2))

共同信息：
{
  "模型来源": "openai",
  "模型": "LongCat-2.0",
  "任务数量": 2
}
消融对照：
[
  {
    "做法": "单步执行",
    "API 调用": 2,
    "等待时间（毫秒）": 4871,
    "最终答案": [
      null,
      null
    ],
    "成功率": "0%"
  },
  {
    "做法": "Agent Loop",
    "API 调用": 4,
    "等待时间（毫秒）": 8898,
    "最终答案": [
      "杭州",
      "42"
    ],
    "成功率": "100%"
  }
]


输出显示两种做法使用相同的真实模型、工具和两个任务。单步执行每项任务只调用一次 API，但工具结果没有进入下一轮，最终答案为空，成功率为 0%；Agent Loop 每项任务调用两次 API，成功提交了“杭州”和“42”，成功率为 100%。模型没有改变，决定任务能否走完的是模型外层有没有循环。